# CS 198/199 
# Joaquin B. Salvador
# Explainable AI for Post-COVID Psychological Profile Prediction

## 1.1
### Scatterplots

In [1]:
# -------------------------------------
# Library configuration
# -------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('..\\data\\interim\\dataset_differences.csv')
df.set_index('SAMPLEID', inplace=True)

In [3]:
# Calculate frequency of each unique metric
frequency = df['K6_total_diff'].value_counts()

# Display the frequency
print("\nFrequency of each metric:")
print(frequency)


Frequency of each metric:
K6_total_diff
 0.0     1080
-1.0      250
 1.0      176
-2.0      146
 2.0      133
-3.0      124
 3.0      101
-4.0       75
-5.0       67
 4.0       59
 5.0       51
-6.0       47
 6.0       45
 7.0       32
-7.0       32
-8.0       24
 12.0      24
-9.0       22
 9.0       20
-10.0      20
 8.0       19
-12.0      14
 10.0      11
-13.0      10
 11.0      10
-11.0      10
 13.0       9
-14.0       7
 14.0       6
 20.0       4
-17.0       4
-23.0       3
 18.0       3
-24.0       3
-19.0       3
 24.0       2
 22.0       2
-16.0       2
 23.0       2
-18.0       2
 19.0       1
 17.0       1
-15.0       1
-20.0       1
-22.0       1
Name: count, dtype: int64


In [4]:
# Function to calculate the required metrics for each column
def summarize_data(df):
    summary = {}
    
    for column in df.columns:
        col_data = df[column]
        summary[column] = {
            'Max': col_data.max(),
            'Min': col_data.min(),
            'Mean': col_data.mean(),
            'Median': col_data.median(),
            'Mode': col_data.mode().values[0] if not col_data.mode().empty else np.nan,
            'Range': col_data.max() - col_data.min(),
            'Standard Deviation': col_data.std(),
            'Variance': col_data.var()
        }
    
    return pd.DataFrame(summary)

In [5]:
# Get the summary
summary_df = summarize_data(df)
print(summary_df.T)  # Transpose to make it easier to read

                        Max    Min       Mean  Median  Mode  Range  \
K6_total_old           24.0   0.00   3.684468     1.0   0.0  24.00   
PHQ9_total_old         27.0   0.00   3.547574     1.0   0.0  27.00   
GAD7_total_old         21.0   0.00   2.570891     1.0   0.0  21.00   
SSS8_total_old         32.0   0.00   4.853328     3.0   0.0  32.00   
PTGI-X(Q7_22_27)_old   36.0   0.00  12.148552    14.0  18.0  36.00   
SHS_total_old           7.0   1.00   4.421023     4.5   4.0   6.00   
UCLA_total_old         40.0  10.00  23.890936    24.0  25.0  30.00   
LSNS6_total_old        30.0   0.00   9.208725     9.0   0.0  30.00   
AUDIT_old              35.0   0.00   4.395637     3.0   0.0  35.00   
K6_total_new           24.0   0.00   3.510342     1.0   0.0  24.00   
PHQ9_total_new         27.0   0.00   3.389996     1.0   0.0  27.00   
GAD7_total_new         21.0   0.00   2.299737     0.0   0.0  21.00   
SSS8_total_new         32.0   0.00   4.696879     3.0   0.0  32.00   
PTGI-X(Q7_22_27)_new

In [ ]:
mh_metrics = [
    "K6_total",
    "PHQ9_total",
    "GAD7_total",
    "SSS8_total",
    "PTGI-X(Q7_22_27)",
    "SHS_total",
    "UCLA_total",
    "LSNS6_total",
    "AUDIT"
]

mh_ranges = [
    [0, 5, 10, 15, 20, 25],            # K6 -> [(0,4), (5,9), (10,14), (15,19), (20,24)]
    [0, 5, 10, 15, 20, 28],            # PHQ9 -> [(0,4), (5,9), (10,14), (15,19), (20,27)]
    [0, 5, 10, 15, 22],                # GAD7 -> [(0,4), (5,9), (10,14), (15,21)]
    [0, 4, 8, 12, 16, 33],             # SS8 -> [(0,3), (4,7), (8,11), (12,15), (16,32)]
    None,                              # PTGI-X
    [1, 2, 3, 4, 5, 6, 7],             # SHS
    [0, 20, 35, 50, 65, 81],           # UCLA -> [(0,19), (20,34), (35,49), (50,64), (65,80)]
    [0, 10, 20, 30, 40, 61],           # LSNS6 -> [(0,9), (10,19), (20,29), (30,39), (40,60)]
    None                               # AUDIT
]

In [ ]:
def scatterplot(mh_metric, mh_range):
    filename = mh_metric + "_scatterplot.png"
    
    plt.figure(figsize=(10, 10))
    
    old = mh_metric + "_old"
    new = mh_metric + "_new"
    scale = mh_metric + "_diff"
    
    # Create hue based on the difference values
    hue = df[scale].apply(lambda x: 'Negative diff' if x < 0 else ('No diff' if x == 0 else 'Positive diff'))
    
    # Calculate counts for the three categories
    counts = hue.value_counts()
    
    # Plotting the scatterplot with hue and adjusted transparency
    sns.scatterplot(x=old, y=new, data=df, hue=hue, 
                    palette={'Negative diff': 'red', 'No diff': 'blue', 'Positive diff': 'green'},
                    alpha=0.3,  # Adjust transparency (0 = fully transparent, 1 = fully opaque)
                    edgecolor='w',  # Optional: add a white edge around points for better separation
                    s=100)  # Optional: increase point size for better visibility
    
    # Add text annotations for counts
    plt.text(1.05, 0.92, f'Positive diff: {counts.get("Positive diff", 0)}', transform=plt.gca().transAxes, 
             fontsize=12, verticalalignment='top', color='green')
    plt.text(1.05, 0.88, f'No diff: {counts.get("No diff", 0)}', transform=plt.gca().transAxes, 
             fontsize=12, verticalalignment='top', color='blue')
    plt.text(1.05, 0.84, f'Negative diff: {counts.get("Negative diff", 0)}', transform=plt.gca().transAxes, 
             fontsize=12, verticalalignment='top', color='red')
    
    # Improve legend position and appearance
    plt.legend(bbox_to_anchor=(1.05, 0.75), loc='upper left', title='Differences', title_fontsize='13', fontsize='10', frameon=True)
    
    plt.title(f'Scatterplot of {mh_metric}')
    plt.xlabel(old)
    plt.ylabel(new)

  # Set dpi for higher resolution
    plt.show()

In [ ]:
i = 0
while i < len(mh_metrics):
    scatterplot(mh_metrics[i], mh_ranges[i])
    i += 1